# 基于 OpenCV 的巡线自动驾驶

在本章教程中我们会在使用 OpenCV 的基础功能来从画面中检测到画面中黄色（默认颜色）的线条，并通过检测该黄色线条的位置来控制底盘转向（本例程中的底盘不会移动，本例程只在画面中展示 OpenCV 的算法），我们这里由于安全方面的原因不会讲运动控制结合在例程里面，因为该功能受外界因素影响比较大，用户需完整理解代码功能后在增加对应的运动控制功能。

如果你想通过本例程来控制机器人移动，请结合前面的 Python 底盘运动控制 章节来添加相关的运动控制函数（我们的开源例程位于 robot_ctrl.py 中）。

## 准备工作
由于产品开机默认会自动运行主程序，主程序会占用摄像头资源，这种情况下是不能使用本教程的，需要结束主程序或禁止主程序自动运行后再重新启动机器人。

这里需要注意的是，由于机器人主程序中使用了多线程且由 service 配置开机自动运行，所以常规的 sudo killall python 的方法通常是不起作用的，所以我们这里介绍禁用主程序自动运行的方法。

如果你已经禁用了机器人主程序的开机自动运行，则不需要执行下面的`结束主程序`章节。

### 结束主程序
1. 点击上方本页面选项卡旁边的 “+”号，会打开一个新的名为 Launcher 的选项卡。
2. 点击 Other 内的 Terminal，打开终端窗口。
3. 在终端窗口内输入 `bash` 后按回车。
4. 现在你可以使用 Bash Shell 来控制机器人了。
5. 输入命令： `systemctl --user stop ugv-app.service`。
6. 在终端页面，命令执行完后，继续该教程的剩余部分。

## 例程
以下代码块可以直接运行：

1. 选中下面的代码块
2. 按 Shift + Enter 运行代码块
3. 观看实时视频窗口
4. 按 `STOP` 关闭实时视频，释放摄像头资源

### 如果运行时不能看到摄像头实时画面
- 需要点击上方的 Kernel - Shut down all kernels
- 关闭本章节选项卡，再次打开
- 点击 `STOP` 释放摄像头资源后重新运行代码块
- 重启设备

### 运行

运行以下代码块后，你可以将黄色胶带放在摄像头前面，观察画面ROI区域中是否有在黄色胶带的轮廓上画点。

In [ ]:
import cv2  # 导入 OpenCV 库，用于图像处理
import imutils, math  # 辅助图像处理和数学运算的库
from picamera2 import Picamera2 # 用于访问 Raspberry Pi Camera 的库
import numpy as np
from IPython.display import display, Image # 用于在 Jupyter Notebook 中显示图像
import ipywidgets as widgets # 用于创建交互式界面的小部件，如按钮
import threading # 用于创建新线程，以便异步执行任务

# Stop button
# ================
stopButton = widgets.ToggleButton(
    value=False,
    description='Stop',
    disabled=False,
    button_style='danger', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Description',
    icon='square' # (FontAwesome names without the `fa-` prefix)
)


# findline autodrive
roi = [
    (250, 300, 40, 600, 0.1),
    (300, 400, 40, 600, 0.3),
    (400, 480, 40, 600, 0.6),
]

# 目标线的颜色，HSV色彩空间
line_lower = np.array([0, 100, 187])
line_upper = np.array([255, 255, 255])

def view(button):
    # 如果你使用的是CSI摄像头 需要取消注释 picam2 这些代码，并注释掉 camera 这些代码
    # 因为新版本的 OpenCV 不再支持 CSI 摄像头（4.9.0.80），你需要使用 picamera2 来获取摄像头画面
    
    # picam2 = Picamera2()  # 创建 Picamera2 的实例
    # 配置摄像头参数，设置视频的格式和大小
    # picam2.configure(picam2.create_video_configuration(main={"format": 'XRGB8888', "size": (640, 480)}))
    # picam2.start()  # 启动摄像头

    camera = cv2.VideoCapture(0) # 创建摄像头实例
    #设置分辨率
    camera.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    camera.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
    
    display_handle=display(None, display_id=True)
    
    while True:
        # img = picam2.capture_array()
        _, img = camera.read() # 从摄像头捕获一帧图像
        # frame = cv2.flip(frame, 1) # if your camera reverses your image

        h, w = img.shape[:2]
        cx = w // 2

        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)

        weight_sum = 0
        cx_sum = 0
        has_line = False

        input_speed = 0.0
        input_turning = 0.0

        for (y1, y2, x1, x2, wt) in roi:
            crop = lab[y1:y2, x1:x2]
            mask = cv2.inRange(crop, line_lower, line_upper)
            cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if not cnts:
                continue

            c = max(cnts, key=cv2.contourArea)
            if cv2.contourArea(c) < 50:
                continue

            (x, y), _, _ = cv2.minAreaRect(c)
            x += x1
            cx_sum += x * wt
            weight_sum += wt
            has_line = True

            cv2.rectangle(img, (x1, y1), (x2, y2), (0,255,0), 1)
            cv2.circle(img, (int(x), int((y1+y2)/2)), 5, (0,0,255), -1)

        if has_line:
            x_avg = int(cx_sum / weight_sum)
            cv2.circle(img, (x_avg, h-50), 8, (0,255,255), -1)

        _, frame = cv2.imencode('.jpeg', img)
        display_handle.update(Image(data=frame.tobytes()))
        if stopButton.value==True:
            # picam2.close()
            camera.release() # 如果是，则关闭摄像头
            display_handle.update(None)


# 显示“停止”按钮并启动显示函数的线程
# ================
display(stopButton)
thread = threading.Thread(target=view, args=(stopButton,))
thread.start()